In [2]:
import cv2
import time

# =========================================================
# SMART VISION - DNN FACE DETECTION
# =========================================================

# =========================================================
# LOAD DNN MODEL
# =========================================================

model_file = "res10_300x300_ssd_iter_140000.caffemodel"
config_file = "deploy.prototxt"

net = cv2.dnn.readNetFromCaffe(
    config_file,
    model_file
)

print("DNN face detector loaded successfully.")


# =========================================================
# CAMERA
# =========================================================

video_cap = cv2.VideoCapture(0)

if not video_cap.isOpened():
    print("ERROR: Camera could not be opened.")
    exit()

video_cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
video_cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

window_name = "Smart Vision"

cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1100, 700)


# =========================================================
# SETTINGS
# =========================================================

CONFIDENCE_THRESHOLD = 0.50
CROWD_LIMIT = 4

previous_time = time.time()
fps = 0

total_crowd_alerts = 0
previous_crowd_state = False


# =========================================================
# MAIN LOOP
# =========================================================

while True:

    ret, frame = video_cap.read()

    if not ret:
        print("ERROR: Could not read camera.")
        break

    current_time = time.time()

    # =====================================================
    # FPS
    # =====================================================

    time_difference = current_time - previous_time

    if time_difference > 0:
        fps = 1 / time_difference

    previous_time = current_time


    # =====================================================
    # PREPARE IMAGE FOR DNN
    # =====================================================

    blob = cv2.dnn.blobFromImage(
        cv2.resize(frame, (300, 300)),
        1.0,
        (300, 300),
        (104.0, 177.0, 123.0)
    )

    net.setInput(blob)

    detections = net.forward()


    # =====================================================
    # DETECT FACES
    # =====================================================

    detected_faces = []

    height, width = frame.shape[:2]

    for i in range(detections.shape[2]):

        confidence = detections[0, 0, i, 2]

        if confidence >= CONFIDENCE_THRESHOLD:

            box = detections[0, 0, i, 3:7]

            x1 = int(box[0] * width)
            y1 = int(box[1] * height)
            x2 = int(box[2] * width)
            y2 = int(box[3] * height)

            # Keep coordinates inside the camera frame
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(width - 1, x2)
            y2 = min(height - 1, y2)

            detected_faces.append(
                (x1, y1, x2, y2, confidence)
            )


    face_count = len(detected_faces)


    # =====================================================
    # CROWD ALERT
    # =====================================================

    crowd_state = face_count >= CROWD_LIMIT

    if crowd_state and not previous_crowd_state:
        total_crowd_alerts += 1

    previous_crowd_state = crowd_state


    # =====================================================
    # DRAW FACE BOXES
    # =====================================================

    for face_number, (x1, y1, x2, y2, confidence) in enumerate(
        detected_faces,
        start=1
    ):

        box_color = (80, 220, 120)

        corner = 18

        # -------------------------------------------------
        # TOP LEFT
        # -------------------------------------------------

        cv2.line(
            frame,
            (x1, y1),
            (x1 + corner, y1),
            box_color,
            3
        )

        cv2.line(
            frame,
            (x1, y1),
            (x1, y1 + corner),
            box_color,
            3
        )

        # -------------------------------------------------
        # TOP RIGHT
        # -------------------------------------------------

        cv2.line(
            frame,
            (x2, y1),
            (x2 - corner, y1),
            box_color,
            3
        )

        cv2.line(
            frame,
            (x2, y1),
            (x2, y1 + corner),
            box_color,
            3
        )

        # -------------------------------------------------
        # BOTTOM LEFT
        # -------------------------------------------------

        cv2.line(
            frame,
            (x1, y2),
            (x1 + corner, y2),
            box_color,
            3
        )

        cv2.line(
            frame,
            (x1, y2),
            (x1, y2 - corner),
            box_color,
            3
        )

        # -------------------------------------------------
        # BOTTOM RIGHT
        # -------------------------------------------------

        cv2.line(
            frame,
            (x2, y2),
            (x2 - corner, y2),
            box_color,
            3
        )

        cv2.line(
            frame,
            (x2, y2),
            (x2, y2 - corner),
            box_color,
            3
        )


        # =================================================
        # FACE NUMBER
        # =================================================

        cv2.putText(
            frame,
            f"FACE {face_number}",
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            2
        )


        # =================================================
        # CONFIDENCE
        # =================================================

        confidence_text = f"{confidence * 100:.0f}%"

        cv2.putText(
            frame,
            confidence_text,
            (x1, y2 + 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            box_color,
            2
        )


    # =====================================================
    # TITLE
    # =====================================================

    cv2.putText(
        frame,
        "SMART VISION",
        (30, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        "DNN FACE ANALYTICS",
        (32, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (180, 190, 200),
        1
    )


    # =====================================================
    # LIVE INDICATOR
    # =====================================================

    cv2.circle(
        frame,
        (frame.shape[1] - 85, 30),
        6,
        (80, 220, 120),
        -1
    )

    cv2.putText(
        frame,
        "LIVE",
        (frame.shape[1] - 70, 36),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2
    )


    # =====================================================
    # FACE COUNT
    # =====================================================

    cv2.putText(
        frame,
        "FACES",
        (30, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (180, 190, 200),
        1
    )

    cv2.putText(
        frame,
        str(face_count),
        (30, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.4,
        (80, 220, 120),
        3
    )


    # =====================================================
    # FPS
    # =====================================================

    cv2.putText(
        frame,
        "FPS",
        (150, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (180, 190, 200),
        1
    )

    cv2.putText(
        frame,
        f"{fps:.0f}",
        (150, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.4,
        (255, 255, 255),
        3
    )


    # =====================================================
    # CROWD ALERTS
    # =====================================================

    cv2.putText(
        frame,
        f"CROWD ALERTS   {total_crowd_alerts}",
        (30, 210),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (190, 200, 210),
        1
    )


    # =====================================================
    # STATUS
    # =====================================================

    if face_count >= CROWD_LIMIT:

        status = "CROWD LEVEL HIGH"
        status_color = (80, 80, 255)

    elif face_count > 0:

        status = "PERSONS DETECTED"
        status_color = (80, 220, 120)

    else:

        status = "AREA CLEAR"
        status_color = (200, 200, 200)


    cv2.putText(
        frame,
        status,
        (frame.shape[1] - 250, frame.shape[0] - 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        status_color,
        2
    )


    # =====================================================
    # CONTROLS
    # =====================================================

    bottom_y = frame.shape[0] - 25

    cv2.putText(
        frame,
        "[F] FULLSCREEN",
        (25, bottom_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (200, 205, 210),
        1
    )

    cv2.putText(
        frame,
        "[S] WINDOW",
        (190, bottom_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (200, 205, 210),
        1
    )

    cv2.putText(
        frame,
        "[C] EXIT",
        (330, bottom_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (200, 205, 210),
        1
    )


    # =====================================================
    # SHOW CAMERA
    # =====================================================

    cv2.imshow(window_name, frame)

    key = cv2.waitKey(1) & 0xFF


    # =====================================================
    # CONTROLS
    # =====================================================

    if key == ord("c"):
        break

    elif key == ord("f"):

        cv2.setWindowProperty(
            window_name,
            cv2.WND_PROP_FULLSCREEN,
            cv2.WINDOW_FULLSCREEN
        )

    elif key == ord("s"):

        cv2.setWindowProperty(
            window_name,
            cv2.WND_PROP_FULLSCREEN,
            cv2.WINDOW_NORMAL
        )

        cv2.resizeWindow(
            window_name,
            1100,
            700
        )


# =========================================================
# CLEANUP
# =========================================================

video_cap.release()
cv2.destroyAllWindows()

DNN face detector loaded successfully.


In [3]:
import cv2
import numpy as np
import os
import glob
import time
from PIL import Image, ImageOps


# ============================================================
# SMART VISION — FACE RECOGNITION
# YuNet + SFace
# ============================================================

YUNET_MODEL = "face_detection_yunet_2023mar.onnx"
SFACE_MODEL = "face_recognition_sface_2021dec.onnx"
KNOWN_FACES_DIR = "known_faces"

CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720

DETECTION_WIDTH = 640
DETECTION_HEIGHT = 360

# Detection threshold
DETECTION_CONFIDENCE = 0.60

NMS_THRESHOLD = 0.3
TOP_K = 5000

# SFace cosine similarity threshold
RECOGNITION_THRESHOLD = 0.40

# Run recognition once every 8 frames
RECOGNITION_INTERVAL = 12


# ============================================================
# CHECK FILES
# ============================================================

for file in [YUNET_MODEL, SFACE_MODEL]:

    if not os.path.exists(file):
        raise FileNotFoundError(
            f"Missing file: {file}"
        )

if not os.path.exists(KNOWN_FACES_DIR):

    raise FileNotFoundError(
        "known_faces folder not found."
    )


# ============================================================
# LOAD MODELS
# ============================================================

print("Loading YuNet detector...")

detector = cv2.FaceDetectorYN.create(
    YUNET_MODEL,
    "",
    (320, 320),
    DETECTION_CONFIDENCE,
    NMS_THRESHOLD,
    TOP_K
)
detector.setInputSize(
    (DETECTION_WIDTH, DETECTION_HEIGHT)
)


print("Loading SFace recognizer...")

recognizer = cv2.FaceRecognizerSF.create(
    SFACE_MODEL,
    ""
)

print("Models loaded successfully.")


# ============================================================
# LOAD KNOWN PEOPLE
# ============================================================

known_features = {}

print("\nLoading known faces...")


person_folders = [
    folder
    for folder in glob.glob(
        os.path.join(KNOWN_FACES_DIR, "*")
    )
    if os.path.isdir(folder)
]


if len(person_folders) == 0:

    raise RuntimeError(
        "No person folders found inside known_faces."
    )


for person_folder in person_folders:

    person_name = os.path.basename(
        person_folder
    )

    # --------------------------------------------------------
    # GET IMAGE FILES
    # --------------------------------------------------------

    image_files = []

    for file in glob.glob(
        os.path.join(person_folder, "*")
    ):

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            image_files.append(file)


    # Remove duplicates
    image_files = list(
        dict.fromkeys(image_files)
    )


    if len(image_files) == 0:

        print(
            f"No images found for {person_name}"
        )

        continue


    person_features = []

    print(
        f"\nPerson: {person_name}"
    )


    # ========================================================
    # PROCESS EACH PHOTO
    # ========================================================

    for image_path in image_files:

        try:

            # Load image
            pil_image = Image.open(
                image_path
            )

            # Correct phone/WhatsApp rotation
            pil_image = ImageOps.exif_transpose(
                pil_image
            )

            pil_image = pil_image.convert(
                "RGB"
            )

            # PIL → OpenCV
            image = cv2.cvtColor(
                np.array(pil_image),
                cv2.COLOR_RGB2BGR
            )


            # Enlarge very small photos
            if (
                image.shape[1] < 500
                or
                image.shape[0] < 500
            ):

                image = cv2.resize(
                    image,
                    None,
                    fx=2.0,
                    fy=2.0,
                    interpolation=cv2.INTER_CUBIC
                )


            # ------------------------------------------------
            # DETECT FACE
            # ------------------------------------------------

            detector.setInputSize(
                (
                    image.shape[1],
                    image.shape[0]
                )
            )

            _, faces = detector.detect(
                image
            )


            if faces is None or len(faces) == 0:

                print(
                    "No face found:",
                    os.path.basename(image_path)
                )

                continue


            # ------------------------------------------------
            # SELECT LARGEST FACE
            # ------------------------------------------------

            largest_face = max(
                faces,
                key=lambda f: f[2] * f[3]
            )


            # ------------------------------------------------
            # ALIGN FACE
            # ------------------------------------------------

            aligned_face = recognizer.alignCrop(
                image,
                largest_face
            )


            # ------------------------------------------------
            # EXTRACT FEATURE
            # ------------------------------------------------

            feature = recognizer.feature(
                aligned_face
            )

            feature = (
                feature /
                np.linalg.norm(feature)
            )


            person_features.append(
                feature
            )


            print(
                "Loaded:",
                os.path.basename(image_path)
            )


        except Exception as e:

            print(
                "Error processing",
                os.path.basename(image_path),
                ":",
                e
            )


    # ========================================================
    # CREATE AVERAGE FEATURE FOR PERSON
    # ========================================================

    if len(person_features) > 0:

        average_feature = np.mean(
            person_features,
            axis=0
        )

        average_feature = (
            average_feature /
            np.linalg.norm(average_feature)
        )

        known_features[
            person_name
        ] = average_feature


        print(
            f"✓ {person_name} registered using "
            f"{len(person_features)} photo(s)"
        )


# ============================================================
# SHOW REGISTERED PEOPLE
# ============================================================

if len(known_features) == 0:

    raise RuntimeError(
        "No usable faces were registered."
    )


print("\n===================================")
print("KNOWN PEOPLE:")

for name in known_features:

    print("✓", name)

print("===================================\n")

detector.setInputSize(
    (DETECTION_WIDTH, DETECTION_HEIGHT)
)



# ============================================================
# CAMERA
# ============================================================

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)

# Request MJPEG, which may help with higher-resolution webcam modes
cap.set(cv2.CAP_PROP_FOURCC,
        cv2.VideoWriter_fourcc(*"MJPG"))

# Request 720p landscape resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# Request 30 FPS (the camera may not accept it)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

print("Backend:", cap.getBackendName())
print("Width:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("FourCC:", int(cap.get(cv2.CAP_PROP_FOURCC)))


if not cap.isOpened():
    print("Camera could not be opened.")
else:
    print("Requested camera resolution: 1280 x 720")
    print("Reported width:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    print("Reported height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    ret, test_frame = cap.read()

    if ret:
        print("Actual camera frame:", test_frame.shape)
    else:
        print("Camera opened, but no frame was received.")


# ============================================================
# WINDOW
# ============================================================

WINDOW_NAME = (
    "Smart Vision - Face Recognition"
)


# ============================================================
# VARIABLES
# ============================================================

frame_count = 0

fps = 0

fps_timer = time.time()

fps_frames = 0

last_names = []

detect_ms = 0.0
recognition_ms = 0.0
loop_ms = 0.0



# ============================================================
# DRAW CORNER BOX
# ============================================================

def draw_corner_box(
    frame,
    x,
    y,
    w,
    h,
    color,
    thickness=2
):

    corner = 18


    # Top-left
    cv2.line(
        frame,
        (x, y),
        (x + corner, y),
        color,
        thickness
    )

    cv2.line(
        frame,
        (x, y),
        (x, y + corner),
        color,
        thickness
    )


    # Top-right
    cv2.line(
        frame,
        (x + w, y),
        (x + w - corner, y),
        color,
        thickness
    )

    cv2.line(
        frame,
        (x + w, y),
        (x + w, y + corner),
        color,
        thickness
    )


    # Bottom-left
    cv2.line(
        frame,
        (x, y + h),
        (x + corner, y + h),
        color,
        thickness
    )

    cv2.line(
        frame,
        (x, y + h),
        (x, y + h - corner),
        color,
        thickness
    )


    # Bottom-right
    cv2.line(
        frame,
        (x + w, y + h),
        (x + w - corner, y + h),
        color,
        thickness
    )

    cv2.line(
        frame,
        (x + w, y + h),
        (x + w, y + h - corner),
        color,
        thickness
    )


# ============================================================
# START
# ============================================================

print("Smart Vision started.")

print("F = Fullscreen")
print("S = Windowed")
print("C = Exit")


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    ret, frame = cap.read()


    if not ret:

        print("Could not read camera.")

        break


    frame_count += 1
    fps_frames += 1


    # --------------------------------------------------------
    # FPS
    # --------------------------------------------------------

    current_time = time.time()


    if current_time - fps_timer >= 1.0:

        fps = (
            fps_frames /
            (current_time - fps_timer)
        )

        fps_frames = 0
        fps_timer = current_time


# --------------------------------------------------------
# FACE DETECTION
# --------------------------------------------------------

    small_frame = cv2.resize(
        frame,
        (DETECTION_WIDTH, DETECTION_HEIGHT),
        interpolation=cv2.INTER_LINEAR
    )


    detector.setInputSize((DETECTION_WIDTH, DETECTION_HEIGHT))
    
    _, small_faces = detector.detect(small_frame)

    scale_x = frame.shape[1] / DETECTION_WIDTH
    scale_y = frame.shape[0] / DETECTION_HEIGHT

    faces = small_faces.copy() if small_faces is not None else None

    if faces is not None:
        faces[:, [0, 2, 4, 6, 8, 10, 12]] *= scale_x
        faces[:, [1, 3, 5, 7, 9, 11, 13]] *= scale_y
    else:
        
        faces = None



    current_names = []


    # --------------------------------------------------------
    # FACE RECOGNITION
    # --------------------------------------------------------

    if faces is not None:

          
        for i, face in enumerate(faces):

            x, y, w, h = (
                face[:4].astype(int)
            )


            # Keep inside frame
            x = max(0, x)
            y = max(0, y)


            w = min(
                w,
                frame.shape[1] - x
            )

            h = min(
                h,
                frame.shape[0] - y
            )


            name = "UNKNOWN"



            # ------------------------------------------------
            # SFACE RECOGNITION
            # ------------------------------------------------
           
            
            
            if (
                frame_count %
                RECOGNITION_INTERVAL == 0
            ):

                try:

                    aligned_face = (
                        recognizer.alignCrop(
                            frame,
                            face
                        )
                    )


                    live_feature = (
                        recognizer.feature(
                            aligned_face
                        )
                    )


                    live_feature = (
                        live_feature /
                        np.linalg.norm(
                            live_feature
                        )
                    )


                    best_score = -1

                    best_name = "UNKNOWN"


                    # Compare with all known people
                    for (
                        person_name,
                        known_feature
                    ) in known_features.items():

                        score = (
                            recognizer.match(
                                known_feature,
                                live_feature,
                                cv2.FaceRecognizerSF_FR_COSINE
                            )
                        )


                        if score > best_score:

                            best_score = score

                            best_name = (
                                person_name
                            )


                    if (
                        best_score >=
                        RECOGNITION_THRESHOLD
                    ):

                        name = best_name

                    else:

                        name = "UNKNOWN"


                except Exception:

                    name = "UNKNOWN"             


            # ------------------------------------------------
            # USE PREVIOUS RESULT
            # ------------------------------------------------
            


            else:

                if i < len(last_names):

                    name = last_names[i]

            current_names.append(
                name
            )
            
            


            # ------------------------------------------------
            # BOX COLOR
            # ------------------------------------------------

            if name == "UNKNOWN":

                box_color = (
                    0,
                    165,
                    255
                )

            else:

                box_color = (
                    0,
                    255,
                    0
                )


            # ------------------------------------------------
            # DRAW BOX
            # ------------------------------------------------

            draw_corner_box(
                frame,
                x,
                y,
                w,
                h,
                box_color,
                2
            )


            # ------------------------------------------------
            # NAME
            # ------------------------------------------------

            label_y = y + h + 28


            if (
                label_y >
                frame.shape[0] - 10
            ):

                label_y = y - 10


            cv2.putText(
                frame,
                name,
                (x, label_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                box_color,
                2,
                cv2.LINE_AA
            )


            # ------------------------------------------------
            # FACE NUMBER + DETECTION SCORE
            # ------------------------------------------------

            detection_score = face[14]


            cv2.putText(
                frame,
                f"FACE {i+1}  "
                f"{detection_score * 100:.0f}%",
                (
                    x,
                    max(20, y - 8)
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                box_color,
                1,
                cv2.LINE_AA
  
            )


    last_names = current_names
    



    # ========================================================
    # HUD
    # ========================================================

    face_count = (
        0
        if faces is None
        else len(faces)
    )


    # Title
    cv2.putText(
        frame,
        "SMART VISION",
        (30, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    cv2.putText(
        frame,
        "AI FACE RECOGNITION",
        (32, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (200, 200, 200),
        1,
        cv2.LINE_AA
    )


    # LIVE
    cv2.circle(
        frame,
        (32, 105),
        6,
        (0, 0, 255),
        -1
    )


    cv2.putText(
        frame,
        "LIVE",
        (45, 111),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # Face count
    cv2.putText(
        frame,
        f"FACES  {face_count}",
        (
            30,
            frame.shape[0] - 65
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # FPS
    cv2.putText(
        frame,
        f"FPS  {fps:.1f}",
        (
            30,
            frame.shape[0] - 35
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # Status
    if face_count == 0:

        status = "SEARCHING..."

    else:

        status = "RECOGNITION ACTIVE"


    cv2.putText(
        frame,
        status,
        (
            frame.shape[1] - 230,
            frame.shape[0] - 35
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (200, 200, 200),
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # DISPLAY + CONTROLS
    # ========================================================


    cv2.imshow(
        WINDOW_NAME,
        frame
    )


    key = cv2.waitKey(1) & 0xFF


    # F = Fullscreen
    if key == ord("f"):

        cv2.setWindowProperty(
            WINDOW_NAME,
            cv2.WND_PROP_FULLSCREEN,
            cv2.WINDOW_FULLSCREEN
        )


    # S = Windowed
    elif key == ord("s"):

        cv2.setWindowProperty(
            WINDOW_NAME,
            cv2.WND_PROP_FULLSCREEN,
            cv2.WINDOW_NORMAL
        )


    # C = Exit
    elif key == ord("c"):

        break


# ============================================================
# CLEANUP
# ============================================================

cap.release()

cv2.destroyAllWindows()

print("Smart Vision stopped.")

Loading YuNet detector...
Loading SFace recognizer...
Models loaded successfully.

Loading known faces...

Person: Prashik Tathode
Loaded: WhatsApp Image 2026-09-30 at 12.53.02 PM (1).jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.53.02 PM (2).jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.53.02 PM.jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.53.03 PM (1).jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.53.03 PM.jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.53.04 PM.jpeg
✓ Prashik Tathode registered using 6 photo(s)

Person: Sahil Talande
Loaded: WhatsApp Image 2026-09-30 at 12.51.09 PM.jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.51.10 PM (1).jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.51.10 PM.jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.51.11 PM (1).jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.51.11 PM.jpeg
Loaded: WhatsApp Image 2026-09-30 at 12.51.13 PM.jpeg
✓ Sahil Talande registered using 6 photo(s)

KNOWN PEOPLE:
✓ Prashik Tathode
✓ Sahil Talande

Backend: MSMF
Width: 1280.0
Height: 72